In [19]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report, 
    roc_auc_score,
    f1_score,
    recall_score
)

In [5]:
df = pd.read_csv("../Data/Preprocessed_data.csv")
df.head()

,person_age,person_income,person_monthly_income,person_emp_length,emp_stability,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,...,person_home_ownership_OTHER,credit_hist_ratio,debt_burden_index,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,3.091042,9.169623,6.685861,1.791759,0.215111,1,6.908755,2.496506,0,0.095310,...,0,0.10,0.113329,1,0,1,0,0,0,0
1,3.258097,9.169623,6.685861,0.693147,0.039221,2,8.612685,2.629728,1,0.451076,...,0,0.12,0.500775,0,0,0,0,1,0,0
2,3.178054,11.089821,8.605081,1.609438,0.157004,2,10.463132,2.786861,1,0.425268,...,0,0.09,0.482426,0,1,0,0,1,0,0
3,3.218876,10.904138,8.419433,2.197225,0.285179,2,10.463132,2.725890,1,0.438255,...,0,0.17,0.553885,0,1,0,0,1,0,0
4,3.091042,9.200391,6.716595,1.098612,0.095310,0,7.824446,2.096790,1,0.223144,...,0,0.10,0.239017,1,0,0,0,0,0,1


In [6]:
x = df.drop(['loan_status'], axis=1)
y = df['loan_status']

In [7]:
xtrain, xtest, ytrain, ytest = train_test_split(
    x, y, test_size=0.2, 
    stratify=y, random_state=42
)

In [8]:
algorithms = {
    "LogisticRegression": LogisticRegression(),
    "RandomForestClassifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier()
}

# model training without scalling

In [9]:
result_list = []

for model_name, models in algorithms.items():
    model = models.fit(xtrain, ytrain)
    y_pred = model.predict(xtest)

    result_list.append({
        'Model': model_name,
        'Train Score': model.score(xtrain, ytrain),
        'Test Score': model.score(xtest, ytest),
        'accuracy_score': accuracy_score(ytest, y_pred),
        'Recall': recall_score(ytest, y_pred),
        'f1_score': f1_score(ytest, y_pred),
        'Confusion Matrix': confusion_matrix(ytest, y_pred)
    })

result = pd.DataFrame(result_list)

In [10]:
print("Without Scalling")
result.sort_values(by='accuracy_score', ascending=False)

Without Scalling


,Model,Train Score,Test Score,accuracy_score,Recall,f1_score,Confusion Matrix
2,XGBClassifier,0.960206,0.937956,0.937956,0.753857,0.839951,"[[4885, 56], [335, 1026]]"
1,RandomForestClassifier,1.000000,0.935576,0.935576,0.734019,0.831115,"[[4897, 44], [362, 999]]"
0,LogisticRegression,0.860266,0.861473,0.861473,0.529023,0.622568,"[[4709, 232], [641, 720]]"


In [11]:
scale_xtrain = xtrain.copy()
scale_xtest = xtest.copy()

In [12]:
scaler = StandardScaler()
scale_xtrain = scaler.fit_transform(scale_xtrain)
scale_xtest = scaler.transform(scale_xtest)

In [13]:
smote = SMOTE()
scale_xtrain_res, scale_ytrain_res = smote.fit_resample(scale_xtrain, ytrain)

In [14]:
result_list_2 = []

for model_name, models in algorithms.items():
    model = models.fit(scale_xtrain_res, scale_ytrain_res)
    y_pred = model.predict(scale_xtest)

    result_list_2.append({
        'Model': model_name,
        'Train Score': model.score(scale_xtrain_res, scale_ytrain_res),
        'Test Score': model.score(scale_xtest, ytest),
        'Accuracy': accuracy_score(ytest, y_pred),
        'Recall': recall_score(ytest, y_pred),
        'f1_score': f1_score(ytest, y_pred),
        'Confusion Matrix': confusion_matrix(ytest, y_pred)
    })

result_2 = pd.DataFrame(result_list_2)

In [15]:
print("With Scalling")
result_2.sort_values(by='Accuracy', ascending=False)

With Scalling


,Model,Train Score,Test Score,Accuracy,Recall,f1_score,Confusion Matrix
2,XGBClassifier,0.969838,0.935893,0.935893,0.749449,0.834697,"[[4878, 63], [341, 1020]]"
1,RandomForestClassifier,1.000000,0.930657,0.930657,0.742836,0.822285,"[[4854, 87], [350, 1011]]"
0,LogisticRegression,0.803973,0.802126,0.802126,0.786921,0.632045,"[[3984, 957], [290, 1071]]"


# Model train & Evaluation Function

In [16]:
def evaluate_model(model, xtrain, xtest, ytrain, ytest, model_name):
    xtrain_pred = model.predict(xtrain)
    y_pred = model.predict(xtest)
    y_pred_proba = model.predict_proba(xtest)[:, 1]

    train_recall = recall_score(ytrain, xtrain_pred)
    test_recall = recall_score(ytest, y_pred)

    accuracay = accuracy_score(ytest, y_pred)
    recall = recall_score(ytest, y_pred)
    f1 = f1_score(ytest, y_pred)
    confusion = confusion_matrix(ytest, y_pred)
    roc_auc = roc_auc_score(ytest, y_pred_proba)

    best_param = model.best_params_ if hasattr(model, 'best_params_') else None
    best_score = model.best_score_ if hasattr(model, 'best_score_') else None

    result = pd.DataFrame([{
        'Model' : model_name,
        'Train Recall' : train_recall,
        'Test Recall' : test_recall,
        'Accuracay' : accuracay,
        'Recall Score' : recall,
        'F1 Score' : f1,
        'Confusion': confusion,
        'Roc Auc': roc_auc,
        'Best Param' : str(best_param),
        'Best Score' : best_score
    }])
    
    return result, y_pred_proba, confusion, roc_auc

In [22]:
xg_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(
        eval_metric = 'logloss',
        random_state = 42, 
        n_jobs = -1
    ))
])

xg_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.05, 0.1],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

xg_cv = StratifiedKFold(
    n_splits=5, 
    shuffle= True,
    random_state=42
)

xg_gridsearchcv = GridSearchCV(
    estimator=xg_pipeline,
    param_grid= xg_param_grid,
    scoring = 'recall',
    cv = xg_cv,
    verbose=1, 
    n_jobs=-1
)

xg_model = xg_gridsearchcv.fit(xtrain, ytrain)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


In [23]:
xg_result, xg_pred_proba, xg_confusion, xg_roc_auc = evaluate_model(
    model=xg_model,
    xtrain=scale_xtrain, 
    xtest=scale_xtest,
    ytrain=ytrain, 
    ytest=ytest,
    model_name='Random Forest Classifier'
)
xg_result

,Model,Train Recall,Test Recall,Accuracay,Recall Score,F1 Score,Confusion,Roc Auc,Best Param,Best Score
0,Random Forest Classifier,0.974472,0.973549,0.325294,0.973549,0.383947,"[[725, 4216], [36, 1325]]",0.711675,"{'model__colsample_bytree': 0.8, 'model__learn...",0.773003
